In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case)

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/10101
10101


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

In [7]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [8]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [9]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1':
    cntrl_vars_init = [1]
elif case[3] == '2':
    cntrl_vars_init = [0,1]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [10]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_' + case + '.pickle'
case_1 = case[0] + case[1] + '0' + case[3] + case[4]
final_file_1 = 'control_' + case_1 + '.pickle'

In [11]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

In [12]:
# get initial parameters and target states

i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
i_range_0 = range(0, len(exc),i_stepsize)
i_range_1 = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [13]:
# get uncontrolled cost

data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    
    if not type(bestControl_init[i]) == type(None):
        continue
        
    control0 = aln.getZeroControl()

    ##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)

    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    with open(init_file,'wb') as f:
        pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  1 , total integrated cost =  37.59293969068204
RUN  2 , total integrated cost =  33.8574829671503
RUN  3 , total integrated cost =  28.464639783237182
RUN  4 , total integrated cost =  26.044325776318136
RUN  5 , total integrated cost =  22.354898110507197
RUN  6 , total integrated cost =  20.5777848079921
RUN  7 , total integrated cost =  17.659723877512086
RUN  8 , total integrated cost =  16.33301825936177
RUN  9 , total integrated cost =  13.721628270481675
RUN  10 , total integrated cost =  12.412346140555654
RUN  11 , total integrated cost =  10.938483046243105
RUN  12 , total integrated cost =  10.91335636909109
RUN  13 , total integrated cost =  10.88167999484503
RUN  14 , total integrated cost =  10.85138500656702
RUN  15 , total integrated cost =  10.748532639225775
RUN  16

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  1962 , total integrated cost =  8.653464659309773
Improved over  1962  iterations in  240.49462509900331  seconds by  99.85339090606945  percent.
Problem in initial value trasfer:  Vmean_exc -62.79057410262651 -62.7893908533989
weight =  6820.859287716994
set cost params:  1.0 0.0 6820.859287716994
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5879.198330319998
Gradient descend method:  None
RUN  1 , total integrated cost =  5814.611932152175
RUN  2 , total integrated cost =  5814.605499660457
RUN  3 , total integrated cost =  5814.605065070712
RUN  4 , total integrated cost =  5814.602335632985
RUN  5 , total integrated cost =  5814.590025559713
RUN  6 , total integrated cost =  5814.587874618544
RUN  7 , total integrated cost =  5814.587430210245
RUN  8 , total integrated cost =  5814.57655768144
RUN  9 , total integrated cost =  5814.554282149306
RUN  10 , total integrated cost =  5814.55288297753
RUN  11 , total integr

ERROR:root:Problem in initial value trasfer


RUN  170 , total integrated cost =  5812.702459123392
Control only changes marginally.
RUN  171 , total integrated cost =  5812.702459123392
Improved over  171  iterations in  20.23123705945909  seconds by  1.131036366874639  percent.
Problem in initial value trasfer:  Vmean_exc -64.75893378354674 -64.76129438792985
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  27.775589623331495
RUN  2 , total integrated cost =  25.261745079514895
RUN  3 , total integrated cost =  18.988144071372055
RUN  4 , total integrated cost =  17.8239169679096
RUN  5 , total integrated cost =  16.04627432925638
RUN  6 , total integrated cost =  15.275637658261797
RUN  7 , total integrated cost =  14.111498082628268
RUN  8 , total integrated cost =  13.527579154008471
RUN  9 , total integrated cost =  12.718839278428062
RUN  1

...

RUN  10 , total integrated cost =  9917.888034152267
Control only changes marginally.
RUN  10 , total integrated cost =  9917.888034152267
Improved over  10  iterations in  1.4778412356972694  seconds by  0.8669278399111704  percent.
Problem in initial value trasfer:  Vmean_exc -67.26687592983066 -67.32877876705213
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  191.5809377818034
RUN  2 , total integrated cost =  122.78712422204399
RUN  3 , total integrated cost =  65.96997354245924
RUN  4 , total integrated cost =  64.33650295769561
RUN  5 , total integrated cost =  63.320582757407074
RUN  6 , total integrated cost =  62.69809996759183
RUN  7 , total integrated cost =  62.24598131603481
RUN  8 , total integrated cost =  61.870319950137045
RUN  9 , total integrated cost =  61.59774668260908
RUN  10 ,

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  304 , total integrated cost =  52.6296826253686
Improved over  304  iterations in  36.91153462231159  seconds by  99.84190567359822  percent.
Problem in initial value trasfer:  Vmean_exc -65.04318076281757 -65.05873726131361
weight =  6325.337681369794
set cost params:  1.0 0.0 6325.337681369794
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32980.16590480063
Gradient descend method:  None
RUN  1 , total integrated cost =  32013.89922279945
RUN  2 , total integrated cost =  31979.868609734505
RUN  3 , total integrated cost =  31979.86592350629
RUN  4 , total integrated cost =  31979.865007447133
RUN  5 , total integrated cost =  31979.860585917035
RUN  6 , total integrated cost =  31979.835340064627
RUN  7 , total integrated cost =  31979.83165358992
RUN  8 , total integrated cost =  31979.830676425467
RUN  9 , total integrated cost =  31979.824810854938
RUN  10 , total integrated cost =  31979.799811538072
RUN  11 , total 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  22 , total integrated cost =  31978.464823926035
Improved over  22  iterations in  3.4472322519868612  seconds by  3.037283328913716  percent.
Problem in initial value trasfer:  Vmean_exc -57.57918207590453 -57.55982758962617


In [15]:
"""
#plot initial guesses
for i in i_range:
    print("---------", i)
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
    plt.show()
"""

'\n#plot initial guesses\nfor i in i_range:\n    print("---------", i)\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n    plt.show()\n'

In [16]:
found_solution = []
no_solution = []
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break


    for i in i_range:
        print("------- ", i, exc[i], inh[i])        

        if np.abs(np.mean(bestState_init[i][0,0,-300:]) - target[i][0,0,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,0,-100:]) - bestState_init[i][0,0,0]) and np.abs(
            np.mean(bestState_init[i][0,1,-300:]) - target[i][0,1,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,1,-100:]) - bestState_init[i][0,1,0]) and np.amin(
            bestState_init[i][0,0,:]) > target[i][0,0,-1] - 5. and np.amin(
            bestState_init[i][0,1,:]) > target[i][0,1,-1] - 5.:
            # and np.amin(bestState_init[i][0,0,:]) > bestState_init[i][0,0,0] - 1.
            #and np.amin(bestState_init[i][0,1,:]) > bestState_init[i][0,1,0] - 1.:
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if len(found_solution) == 0:
        #    continue
            
        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)

------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
[0] []
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8.518851945280366
Gradient descend method:  None
RUN  1 , total integrated cost =  5.979841577400071
RUN  2 , total integrated cost =  5.9142656186097895
RUN  3 , total integrated cost =  5.913812707585442
RUN  4 , total integrated cost =  5.9137531817303985
RUN  5 , total integrated cost =  5.9135844745763695
RUN  6 , total integrated cost =  5.913472119935345
RUN  7 , total integrated cost =  5.913039876275756
RUN  8 , total integrated cost =  5.912629281584199
RUN  9 , total integrated cost =  5.9121528317271554
RUN  10 , total integrated 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  116 , total integrated cost =  5.863172028559094
Improved over  116  iterations in  15.721890732645988  seconds by  31.1741527353645  percent.
Problem in initial value trasfer:  Vmean_exc -67.8943415408545 -67.89737283779931
weight =  8693.740868204422
set cost params:  1.0 0.0 8693.740868204422
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5092.361083950799
Gradient descend method:  None
RUN  1 , total integrated cost =  5067.694215208252
RUN  2 , total integrated cost =  5067.692007141108
RUN  3 , total integrated cost =  5067.692003194149
RUN  4 , total integrated cost =  5067.692002975701
RUN  5 , total integrated cost =  5067.692002941891
RUN  6 , total integrated cost =  5067.692002937206
RUN  7 , total integrated cost =  5067.692002936622
RUN  8 , total integrated cost =  5067.692002936524
RUN  9 , total integrated cost =  5067.69200293651
RUN  10 , total integrated cost =  5067.692002936488
RUN  11 , total integrat

ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  5067.692002936483
Control only changes marginally.
RUN  13 , total integrated cost =  5067.692002936483
Improved over  13  iterations in  2.359593540430069  seconds by  0.48443306764053773  percent.
Problem in initial value trasfer:  Vmean_exc -66.91387323056234 -66.93439589141886
-------  10 0.4250000000000001 0.42500000000000016
[0] []
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9112.231758647074
Gradient descend method:  None
RUN  1 , total integrated cost =  64.79474455418661
RUN  2 , total integrated cost =  56.07564467809322
RUN  3 , total integrated cost =  40.49806816381058
RUN  4 , total integrated cost =  34.90870565516642
RUN  5 , total integrated cost =  28.27730456989072
RUN  6 , total integrated cost =  25.843871397181015
RUN  7 , total integrated cost =  20.897750410172904
RUN  8 , total integrated cost =  20.76009535323
RUN  9 , total inte

...

RUN  11 , total integrated cost =  9911.221114692795
Control only changes marginally.
RUN  11 , total integrated cost =  9911.221114692795
Improved over  11  iterations in  1.6191612277179956  seconds by  0.9230563666101972  percent.
Problem in initial value trasfer:  Vmean_exc -67.11617031874135 -67.17837965954294
-------  145 0.5750000000000002 0.9000000000000006
[0] []
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33262.256247692385
Gradient descend method:  None
RUN  1 , total integrated cost =  191.95642534655244
RUN  2 , total integrated cost =  122.38424040668528
RUN  3 , total integrated cost =  66.68246312049308
RUN  4 , total integrated cost =  64.3858054511965
RUN  5 , total integrated cost =  61.922851622003094
RUN  6 , total integrated cost =  61.22059094994877
RUN  7 , total integrated cost =  61.05493702286238
RUN  8 , total integrated cost =  60.88012131089138
RUN  9 , total 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  209 , total integrated cost =  52.740712207641536
Improved over  209  iterations in  26.21755526214838  seconds by  99.84143976339158  percent.
Problem in initial value trasfer:  Vmean_exc -65.01200738596722 -65.02763666508226
weight =  6312.021600279862
set cost params:  1.0 0.0 6312.021600279862
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32960.68456717204
Gradient descend method:  None
RUN  1 , total integrated cost =  31935.55929509603
RUN  2 , total integrated cost =  31935.03997260085
RUN  3 , total integrated cost =  31934.681464136003
RUN  4 , total integrated cost =  31934.19718230053
RUN  5 , total integrated cost =  31933.793698760415
RUN  6 , total integrated cost =  31933.149209979365
RUN  7 , total integrated cost =  31932.561017481723
RUN  8 , total integrated cost =  31931.56193145964
RUN  9 , total integrated cost =  31930.66429423111
RUN  10 , total integrated cost =  31925.639795084277
RUN  11 , total 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  56 , total integrated cost =  31898.28079770152
Improved over  56  iterations in  7.386828536167741  seconds by  3.223245461742138  percent.
Problem in initial value trasfer:  Vmean_exc -57.50832207960205 -57.48839410100991
------------------------------------------------------------
-------------------- 1
------------------------------------------------------------
found solution:  [0]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
closest index  -1
set cost params:  1.0 0.0 10.0
all options tried already
-------  10 0.4250000000000001 0.42500000000000016
closest index  -1
set cost params:  1.0 0.0 10.0
all options tried already
-------  15 0.4500000000000001 0.4500000000000002
closest index  -1
set cost params:  1.0 0.0 10.0
all options tried already
-------  20 0.4500000000000001 0.4750000000000002
closest index  -1
set cost params:  1.0 0.0 10.0
all options tried already
-----

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print("------------------------------------------------")
    print('-------------------------', counter)
    
    if counter > 20:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_init[i] == [True, True]:
            continue
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
                       
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
    counter += 1

------------------------------------------------
------------------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  6925.121599533168
set cost params:  1.0 0.0 6925.121599533168
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5901.521978081638
Gradient descend method:  None
RUN  1 , total integrated cost =  5901.52196790264
RUN  2 , total integrated cost =  5901.521963463343
RUN  3 , total integrated cost =  5901.521953190246
RUN  4 , total integrated cost =  5901.521953

ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  5901.521953190226
Control only changes marginally.
RUN  6 , total integrated cost =  5901.521953190226
Improved over  6  iterations in  1.2479251120239496  seconds by  4.2177953218924813e-07  percent.
Problem in initial value trasfer:  Vmean_exc -64.75824073426219 -64.76069172199922
no convergence
-------  5 0.4000000000000001 0.40000000000000013
weight =  8743.516610485503
set cost params:  1.0 0.0 8743.516610485503
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5096.6797280210185
Gradient descend method:  None
RUN  1 , total integrated cost =  5096.679609462816
RUN  2 , total integrated cost =  5096.679607201512
RUN  3 , total integrated cost =  5096.679606728859
RUN  4 , total integrated cost =  5096.679606694562
RUN  5 , total integrated cost =  5096.679606693593
RUN  6 , total integrated cost =  5096.6796066935685
RUN  7 , total integrated cost =  5096.679606693562
RUN  8 , total integrated cost =  5096.6796066935585
RUN  

ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  5096.679606693556
Control only changes marginally.
RUN  11 , total integrated cost =  5096.679606693556
Improved over  11  iterations in  2.0712684374302626  seconds by  2.3805196605053425e-06  percent.
Problem in initial value trasfer:  Vmean_exc -66.90746815679267 -66.92810371730843
no convergence
-------  10 0.4250000000000001 0.42500000000000016
weight =  6175.355582416906
set cost params:  1.0 0.0 6175.355582416906
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9109.831610227331
Gradient descend method:  None
RUN  1 , total integrated cost =  9109.83127676086
RUN  2 , total integrated cost =  9109.831264221202
RUN  3 , total integrated cost =  9109.831262371357
RUN  4 , total integrated cost =  9109.831258654118
RUN  5 , total integrated cost =  9109.831256782569
RUN  6 , total integrated cost =  9109.831256782563
RUN  7 , total integrated cost =  9109.831256782554
RUN  8 , total integrated cost =  9109.83125678255
RUN  9

...

RUN  8 , total integrated cost =  34456.628603734076
Control only changes marginally.
RUN  8 , total integrated cost =  34456.628603734076
Improved over  8  iterations in  1.3400187604129314  seconds by  0.00011825804740794865  percent.
Problem in initial value trasfer:  Vmean_exc -56.70385032122269 -56.7037934949543
no convergence
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
weight =  8972.766236035475
set cost params:  1.0 0.0 8972.766236035475
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  39295.585873685406
Gradient descend method:  None
RUN  1 , total integrated cost =  39295.52016451702


ERROR:root:Problem in initial value trasfer


RUN  2 , total integrated cost =  39295.52016451702
Control only changes marginally.
RUN  2 , total integrated cost =  39295.52016451702
Improved over  2  iterations in  0.5014200005680323  seconds by  0.00016721768344041266  percent.
Problem in initial value trasfer:  Vmean_exc -56.701270312717305 -56.70111674246322
no convergence
-------  95 0.5250000000000001 0.7500000000000004
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
-------  130 0.6000000000000003 0.8500000000000005
weight =  8939.79161826293
set cost params:  1.0 0.0 8939.79161826293
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  38683.121465624165
Gradient descend method:  None
RUN  1 , total integrated cost =  38682.961928176366
RUN  2 , tota

ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  38682.96169165159
Control only changes marginally.
RUN  4 , total integrated cost =  38682.96169165159
Improved over  4  iterations in  0.8620672542601824  seconds by  0.00041303278153748124  percent.
Problem in initial value trasfer:  Vmean_exc -56.70162871081352 -56.70149081732219
no convergence
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
------------------------------------------------
------------------------- 21


In [18]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

In [19]:
factor_iteration = 20
conv_0 = [[False]*2] * len(exc)
full_converge = False

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 20:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break
        
    counter += 1
    
    for i in i_range_0:
        print("------- ", i, exc[i], inh[i])
        
        if conv_0[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

    # exc and inh control current 

        setinit(initVars[i], aln)
        aln.params.duration = dur

        if not type(bestControl_0[i]) == type(None):
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
        else:
            control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
            weights_0[i] = weights_init[i]
            cost_0[i] = cost_init[i]

        cgv = None
        max_it = 500 * factor_iteration

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                           + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        with open(final_file,'wb') as f:
            pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                     costnode_0, weights_0], f)
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  6925.159557461547
set cost params:  1.0 0.0 6925.159557461547
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5901.554288867908
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5901.554288867908
Control only changes marginally.
RUN  1 , total integrated cost =  5901.554288867908
Improved over  1  iterations in  1.0267390310764313  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -64.75824073426219 -64.76069172199922
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
weight =  8743.563509159707
set cost params:  1.0 0.0 8743.563509159707
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5096.7069186213475
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5096.7069186213475
Control only changes marginally.
RUN  1 , total integrated cost =  5096.7069186213475
Improved over  1  iterations in  0.9445041120052338  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -66.90746815679267 -66.92810371730843
converged for  5
-------  10 0.4250000000000001 0.42500000000000016
weight =  6175.457449732188
set cost params:  1.0 0.0 6175.457449732188
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9109.98129887924
Gradient descend method:  None


...

RUN  7 , total integrated cost =  34482.83949440757
Control only changes marginally.
RUN  7 , total integrated cost =  34482.83949440757
Improved over  7  iterations in  5.092192359268665  seconds by  8.142795067556108e-06  percent.
Problem in initial value trasfer:  Vmean_exc -56.70355789281839 -56.70351798532243
no convergence
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
weight =  9078.048657944928
set cost params:  1.0 0.0 9078.048657944928
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  39325.91050844944
Gradient descend method:  None
RUN  1 , total integrated cost =  39325.90710181708
RUN  2 , total integrated cost =  39325.9070939667
RUN  3 , total integrated cost =  39325.907093966685


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  39325.907093966685
Control only changes marginally.
RUN  4 , total integrated cost =  39325.907093966685
Improved over  4  iterations in  3.459488783031702  seconds by  8.682526882353159e-06  percent.
Problem in initial value trasfer:  Vmean_exc -56.70061316493642 -56.70051474380661
no convergence
-------  95 0.5250000000000001 0.7500000000000004
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
-------  130 0.6000000000000003 0.8500000000000005
weight =  9044.243323288538
set cost params:  1.0 0.0 9044.243323288538
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  38712.700774534205
Gradient descend method:  None
RUN  1 , total integrated cost =  38712.69704973614
RUN  2 , tota

ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  38712.6970497361
Control only changes marginally.
RUN  3 , total integrated cost =  38712.6970497361
Improved over  3  iterations in  2.542473416775465  seconds by  9.62164361340001e-06  percent.
Problem in initial value trasfer:  Vmean_exc -56.70109855566186 -56.70100947503302
no convergence
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
--------------- 21


In [20]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [21]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

In [22]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    if counter > 20:
        break
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_1[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        if not type(bestControl_1[i]) == type(None):
            control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1].copy()
        else:
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1].copy()
            cost_1[i] = cost_0[i]
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(final_file_1,'wb') as f:
            pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)
            
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
    counter += 1

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.987079390586786
Gradient descend method:  None
RUN  1 , total integrated cost =  0.8832611879539115
RUN  2 , total integrated cost =  0.8830407285020343
RUN  3 , total integrated cost =  0.8826670815230594
RUN  4 , total integrated cost =  0.8823348932612892
RUN  5 , total integrated cost =  0.8814269362198565
RUN  6 , total integrated cost =  0.

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  220 , total integrated cost =  0.8668581552007733
Improved over  220  iterations in  28.032592304050922  seconds by  70.97974168572445  percent.
Problem in initial value trasfer:  Vmean_exc -62.96885054402816 -62.96918782439146
no convergence
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5.384672611378334
Gradient descend method:  None
RUN  1 , total integrated cost =  0.5878606812547156
RUN  2 , total integrated cost =  0.5878453347141216
RUN  3 , total integrated cost =  0.5878452022070996
RUN  4 , total integrated cost =  0.5878451806725795
RUN  5 , total integrated cost =  0.5878451695498172
RUN  6 , total integrated cost =  0.5878451527741688
RUN  7 , total integrated cost =  0.5878450877551624
RUN  8 , total integrated cost =  0.5877295873701139
RUN  9 , total integrated cost =  0.5876458588708469
RUN  10 , total integrated cost =  0.5876

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  126 , total integrated cost =  0.58679188066137
Improved over  126  iterations in  15.903671620413661  seconds by  89.10255231819625  percent.
Problem in initial value trasfer:  Vmean_exc -67.98750769641738 -67.99002475319679
no convergence
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15.518181432172781
Gradient descend method:  None
RUN  1 , total integrated cost =  1.4994578216671908
RUN  2 , total integrated cost =  1.4993845907760246
RUN  3 , total integrated cost =  1.499383926291798
RUN  4 , total integrated cost =  1.4993836318046425
RUN  5 , total integrated cost =  1.49936889765875
RUN  6 , total integrated cost =  1.4993383974855006
RUN  7 , total integrated cost =  1.4993367423166168
RUN  8 , total integrated cost =  1.4993364695679532
RUN  9 , total integrated cost =  1.499336135509309
RUN  10 , total integrated cost =  1.49554435

...

RUN  1 , total integrated cost =  3.903894332693535
Control only changes marginally.
RUN  1 , total integrated cost =  3.903894332693535
Improved over  1  iterations in  0.32693295925855637  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -69.35181534009256 -69.3720519731705
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.5551489907578337
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.5551489907578337
Control only changes marginally.
RUN  1 , total integrated cost =  1.5551489907578337
Improved over  1  iterations in  0.3578691389411688  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -74.45835667428726 -74.49009406143874
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5.297521709179365
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5.297521709179365
Control only changes marginally.
RUN  1 , total integrated cost =  5.297521709179365
Improved over  1  iterations in  0.32247496768832207  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -66.46092377635367 -66.47168426453099
converged for  145
--------------- 3
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
full convergence
